Start simple → Linear Regression with a few features


Try Decision Tree → see overfitting on train/test


Upgrade → Random Forest


Optional → Gradient Boosting


Distance-based → KNN / SVR (requires scaling)

In [1]:
import numpy as np
import pandas as pd

# load features
x_train_scaled = pd.read_csv('x_train_scaled.csv')
x_test_scaled = pd.read_csv('x_test_scaled.csv')

y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

# this changes the dataframe from 2d to 1d, which is what the model expects
#  from (n,1) to (n,) only prices in an array
y_train = y_train.squeeze()
y_test = y_test.squeeze()



In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import RFE

model_lr = LinearRegression()

# rfe is recursive feature elimination, it will select the top 5 features based on the model_lr
rfe = RFE(estimator=model_lr, n_features_to_select=5)
rfe.fit(x_train_scaled, y_train)


,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).",LinearRegression()
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",5
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [3]:
# listing the features, whether they were selected, and their ranking
list(zip(x_train_scaled.columns,rfe.support_,rfe.ranking_))


[('area', np.True_, np.int64(1)),
 ('bedrooms', np.False_, np.int64(7)),
 ('bathrooms', np.True_, np.int64(1)),
 ('stories', np.True_, np.int64(1)),
 ('mainroad', np.False_, np.int64(2)),
 ('guestroom', np.False_, np.int64(3)),
 ('basement', np.False_, np.int64(10)),
 ('hotwaterheating', np.False_, np.int64(4)),
 ('airconditioning', np.True_, np.int64(1)),
 ('parking', np.False_, np.int64(5)),
 ('prefarea', np.False_, np.int64(6)),
 ('furnishingstatus_furnished', np.False_, np.int64(11)),
 ('furnishingstatus_semi-furnished', np.False_, np.int64(9)),
 ('furnishingstatus_unfurnished', np.False_, np.int64(8)),
 ('price_per_area', np.True_, np.int64(1))]

In [4]:
#  listing the features that were selected
col = x_train_scaled.columns[rfe.support_]


In [5]:
#  listing the features that were not selected
x_train_scaled.columns[~rfe.support_]


Index(['bedrooms', 'mainroad', 'guestroom', 'basement', 'hotwaterheating',
       'parking', 'prefarea', 'furnishingstatus_furnished',
       'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished'],
      dtype='str')

In [6]:
# creating new dataframes with only the selected features
x_train_rfe = x_train_scaled[col]
x_test_rfe = x_test_scaled[col]


In [7]:
import statsmodels.api as sm  
# adding a constant to the model (intercept)
x_train_rfe = sm.add_constant(x_train_rfe)
x_test_rfe = sm.add_constant(x_test_rfe)


In [8]:
# fitting the model with the selected features
# Fits linear regression using: Ordinary Least Squares (minimizes squared error)
model_lr = sm.OLS(y_train, x_train_rfe).fit()
print(model_lr.summary())


                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.859
Model:                            OLS   Adj. R-squared:                  0.858
Method:                 Least Squares   F-statistic:                     525.1
Date:                Mon, 30 Mar 2026   Prob (F-statistic):          1.51e-180
Time:                        15:15:07   Log-Likelihood:                -6437.5
No. Observations:                 436   AIC:                         1.289e+04
Df Residuals:                     430   BIC:                         1.291e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const           -4.793e+05   1.27e+05     

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

#  variance inflation factor measures how features are correlated
# high correlarion between features can cause problems with the model, it can make it difficult to interpret the coefficients and can also lead to overfitting
vif = pd.DataFrame()
X = x_train_rfe
vif['Features'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by = "VIF", ascending = False)
vif


,Features,VIF
0,const,17.65
5,price_per_area,1.91
1,area,1.83
2,bathrooms,1.32
3,stories,1.28
4,airconditioning,1.24


In [10]:
y_pred_train = model_lr.predict(x_train_rfe)
y_pred_test = model_lr.predict(x_test_rfe)


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = y_test
y_pred = y_pred_test

rmse = mean_squared_error(y_true, y_pred) ** 0.5
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
max_error = max(abs(y_true - y_pred))
mape = (abs(y_true - y_pred)/y_true).mean() * 100

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R2: {r2}")
print(f"MAPE: {mape}%")
print(f"Max Error: {max_error}")


RMSE: 644544.6426165377
MAE: 487724.18441207596
R2: 0.8971661101386468
MAPE: 10.535488775557841%
Max Error: 2478265.936341917
